# 04_common_track — 논문 라우터 공통 트랙 (EV-10)

[최종 실험 계획서](../../../docs/ml/ROUTER_FINAL_EXPERIMENT_PLAN.md) 4.1·4.3절. 같은 BGE 임베딩(01_embed)과 같은 학습 행(03_ours의 섞은 데이터, kNN-A는 A 분할)으로 kNN·MLP·SVM(RouterBench), MF·GraphRouter·Hybrid LLM(LLMRouter), IRT-Router, Avengers-Pro, R1-BGE(우리)를 학습한다. 집단 셋을 가상의 모델 3개로 보고, 성능은 집단 풀 확률이다.

- 외부 코드는 저장소에 넣지 않는다. 고정 커밋에서 필요한 파일만 내려받고 파일마다 SHA-256을 확인한 뒤 원본 클래스를 그대로 불러 쓴다. API·원격 임베딩·파일 경로 부분만 BGE 행렬로 바꾼다(바꾼 내용은 각 결과의 `deviations`).
- 방법마다 설정 ≤ 8개를 모두 학습하고 dev·test·RouterArena 질문의 단계별 점수를 저장한다. 설정 선택과 보정은 07_lock이 dev에서 한다. test 정답은 읽지 않는다.

**런타임: 2026.07(Python 3.12), T4 GPU.** 예상 약 1.5~2시간(GraphRouter 공식 구조를 빼면 로컬 CPU 전체 실행이 17분이었다. Colab CPU는 2코어라 sklearn 부분이 더 느리고, GraphRouter 공식 구조 1000 epoch 두 번은 GPU에서 수십 분으로 추정한다). 결과는 Drive `common/<방법>/<설정>/`에 설정마다 바로 저장된다. 연결이 끊기면 같은 셀을 다시 실행하면 끝난 설정은 건너뛰고 이어서 한다.

**GraphRouter:** LLMRouter 판(`impl=llmrouter`)은 우리 데이터에서 점수가 모두 0이 되어 `degenerate=True`로 나오는 것이 정상이다(재현 실패 기록). 공식 구조 판(`impl=official`)은 실제 모델을 노드로 두고 끝에서 3단계로 묶는다. 모델별 성공 기록 `run001/common-inputs/`가 필요하며, 로컬 `build-outcomes`로 미리 만들어 Drive에 올려 두었다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import hashlib, json, subprocess, sys, zipfile
RUN = Path('/content/drive/MyDrive/LLM_Router/router-final/runs/run001')
SOURCE_ZIP = RUN / 'code' / 'router-final-source-common.zip'
expected = SOURCE_ZIP.with_suffix('.sha256').read_text().split()[0]
assert hashlib.sha256(SOURCE_ZIP.read_bytes()).hexdigest() == expected, '소스 ZIP이 바뀌었다'
REPO = Path('/content/router-final-source')
with zipfile.ZipFile(SOURCE_ZIP) as z:
    for name in z.namelist():
        assert (REPO / name).resolve().is_relative_to(REPO.resolve()), name
    z.extractall(REPO)
for name, digest in json.loads((REPO / 'source-bundle-manifest.json').read_text()).items():
    assert hashlib.sha256((REPO / name).read_bytes()).hexdigest() == digest, name
assert sys.version_info[:2] in {(3, 11), (3, 12)}, '런타임 버전을 2026.07(Python 3.12)로 바꾸세요'
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchvision', 'torchaudio'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', 'torch==2.5.1', '--index-url', 'https://download.pytorch.org/whl/cu124'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(REPO / 'ml/notebooks/router_final/requirements-common-colab.txt')], check=True)

설치가 끝나면 **런타임 → 세션 다시 시작**을 한 번 한 뒤 아래 셀부터 실행합니다.

In [ ]:
from pathlib import Path
import json, shutil, subprocess, sys
RUN = Path('/content/drive/MyDrive/LLM_Router/router-final/runs/run001')
REPO = Path('/content/router-final-source')
SCRIPT = REPO / 'ml/notebooks/router_final/common_track.py'
EXTERNAL = Path('/content/ev10-external')
FINAL = RUN / 'common'
assert (RUN / 'common-inputs' / 'manifest.json').exists(), 'common-inputs가 없다(로컬 build-outcomes 결과)'
import torch
assert torch.cuda.is_available(), 'T4 GPU 런타임을 선택하세요'
print(torch.cuda.get_device_name(0), torch.__version__)
# 외부 코드: 고정 커밋에서 내려받고 파일별 SHA-256 확인
subprocess.run([sys.executable, str(SCRIPT), 'fetch', '--external-dir', str(EXTERNAL)], check=True)

In [ ]:
# 1) 시험 실행: 원천마다 학습 600행, 역할마다 점수 200행 (몇 분)
shutil.rmtree('/content/common-smoke', ignore_errors=True)
subprocess.run([sys.executable, str(SCRIPT), 'run', '--run-dir', str(RUN), '--external-dir', str(EXTERNAL),
                '--output-dir', '/content/common-smoke', '--smoke', '--device', 'cuda'], check=True)

In [ ]:
# 2) 본 실행 → Drive common/ 에 설정마다 바로 저장. 끊기면 이 셀만 다시 실행
subprocess.run([sys.executable, str(SCRIPT), 'run', '--run-dir', str(RUN), '--external-dir', str(EXTERNAL),
                '--output-dir', str(FINAL), '--device', 'cuda'], check=True)

In [ ]:
# 3) 요약: 완료 여부, 설정별 학습·점수 시간, 퇴화(점수가 상수) 여부
manifest = json.loads((FINAL / 'manifest.json').read_text())
print('complete:', manifest['complete'], '| versions:', manifest['versions'])
for key in manifest['outputs']:
    m = json.loads((FINAL / key / 'metrics.json').read_text())
    print(f"{key:55s} fit {m['fit_seconds']:7.0f}s  score {m['score_seconds']:5.0f}s  degenerate={m['degenerate']}")